# Build a Coding Agent

**What you will build:** a working coding agent — the same basic pattern behind Claude Code, OpenAI's Codex CLI, and OpenCode — in about a hundred lines on top of the loop you wrote in 0.3. It reads and edits files, runs shell commands, asks your permission before destructive actions, and works through a failing test suite.

This chapter ties Block 0 together. The loop (0.3), structured tool calls (0.2), error feedback (0.3), and reflection against a check (0.5) now become a coding-agent pattern you can inspect.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/07_build_a_coding_agent.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1, plus pytest for the verification loop.
!pip install -q openai pytest

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"   # change to any model on openrouter.ai/models
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready. Model:", MODEL)

## You already built the hard part

Thorsten Ball, who works on a commercial coding agent, wrote the canonical post on this ([How to Build an Agent](https://ampcode.com/how-to-build-an-agent)). His conclusion after building one in ~300 lines of Go:

> "It's an LLM, a loop, and enough tokens. The rest … elbow grease."

His agent has exactly three tools: read a file, list files, edit a file. The loop is the one from 0.3. What turns that loop into Claude Code is not a smarter loop — it's the engineering *around* it:

```
                 ┌──────────────────────────────────────────────┐
                 │              THE 0.3 LOOP (the 5%)           │
                 │        while tool_calls: run, feed back      │
                 └──────────────────────────────────────────────┘
   what this notebook adds (the useful 95%):
   ┌──────────────┐ ┌──────────────┐ ┌──────────────┐ ┌──────────────┐
   │  FILE TOOLS  │ │    SHELL     │ │  PERMISSION  │ │ ENVIRONMENT  │
   │ read / list  │ │ run commands │ │     GATE     │ │   CONTEXT    │
   │   / edit     │ │ + timeouts   │ │ ask before   │ │ file tree,   │
   │              │ │ + truncation │ │ writes       │ │ conventions  │
   └──────────────┘ └──────────────┘ └──────────────┘ └──────────────┘
                              plus the one that matters most:
                 ┌──────────────────────────────────────────────┐
                 │   A CHECK THE AGENT CAN RUN (the test suite)  │
                 └──────────────────────────────────────────────┘
```

That last box is the deep idea. David Crawshaw (Sketch) puts it as: *agents are feedback-driven LLMs* — the value comes from the compiler, the tests, and the linter acting as ground truth the model can iterate against. Anthropic's own guidance for Claude Code says the same thing: **give the agent a check it can run.** We will.

## A workspace to work on

The agent needs code to work *on*. We create a tiny sandboxed project: a stats module with a **deliberate bug**, and a pytest suite that catches it. Keeping the agent inside this directory is our first safety decision — every tool below resolves paths relative to `WORKSPACE` and nothing else.

In [ ]:
from pathlib import Path

WORKSPACE = Path("workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

(WORKSPACE / "stats.py").write_text('''def mean(numbers):
    """Arithmetic mean of a list of numbers."""
    return sum(numbers) / len(numbers)


def median(numbers):
    """Median of a list of numbers."""
    ordered = sorted(numbers)
    return ordered[len(ordered) // 2]
''')

(WORKSPACE / "test_stats.py").write_text('''from stats import mean, median


def test_mean():
    assert mean([1, 2, 3, 4]) == 2.5


def test_median_odd():
    assert median([3, 1, 2]) == 2


def test_median_even():
    # the median of an even-length list is the average of the two middle values
    assert median([1, 2, 3, 4]) == 2.5
''')

(WORKSPACE / "AGENTS.md").write_text('''# Project conventions
- Prefer the standard library; do not add dependencies.
- Keep functions small and pure.
- Do not delete or rename existing functions.
''')

print("workspace ready:")
for p in sorted(WORKSPACE.iterdir()):
    print(" ", p.name)

Spot the bug before the agent does: `median` indexes the middle element, which is correct for odd-length lists but wrong for even-length ones (the median of `[1, 2, 3, 4]` is `2.5`, not `3`). The third test will fail. That failing test is not an obstacle — it's the **ground truth** the agent will iterate against.

The `AGENTS.md` file is a convention borrowed directly from the real tools: Claude Code injects a `CLAUDE.md` from your repo into every session, Codex CLI and OpenCode read `AGENTS.md`. It's how a project tells the agent things inference can't guess — style rules, forbidden moves, how to run the tests. Ours is three lines; real ones grow with every correction you find yourself repeating.

## The tools, part 1 — files

Three file tools, modeled on the real ones. Two design decisions are worth reading slowly, because production agents live or die by them:

**Reads are bounded.** `read_file` returns at most `limit` lines, with line numbers, from `offset`. Claude Code caps reads at 2000 lines for the same reason: a tool result goes straight into the context window, and one careless read of a giant file can eat the whole token budget (0.6's lesson, applied).

**Edits demand a unique anchor.** `edit_file` replaces `old_str` with `new_str` only if `old_str` appears **exactly once**. If it matches twice, the edit is refused and the error tells the model to include more surrounding context. This is precisely how Claude Code's Edit tool and Ball's ~300-line agent work — an ambiguous anchor means you might edit the *wrong* occurrence, and a refused edit is infinitely cheaper than a wrong one.

In [ ]:
def read_file(path: str, offset: int = 1, limit: int = 200) -> str:
    """Read a file with line numbers, `limit` lines starting at line `offset`."""
    p = (WORKSPACE / path)
    if not p.is_file():
        return f"Error: {path} does not exist"
    lines = p.read_text().splitlines()
    chunk = lines[offset - 1 : offset - 1 + limit]
    out = "\n".join(f"{n:4d}| {line}" for n, line in enumerate(chunk, start=offset))
    remaining = len(lines) - (offset - 1 + len(chunk))
    if remaining > 0:
        out += f"\n... ({remaining} more lines — call again with offset={offset + len(chunk)})"
    return out or "(empty file)"


IGNORED_DIRS = {".pytest_cache", "__pycache__", ".git"}   # caches are noise to the model —
                                                          # real agents respect .gitignore for the same reason

def list_files(path: str = ".") -> str:
    """List every file in the workspace (relative paths)."""
    base = WORKSPACE / path
    if not base.exists():
        return f"Error: {path} does not exist"
    entries = [str(p.relative_to(WORKSPACE)) for p in sorted(base.rglob("*"))
               if p.is_file() and not any(part in IGNORED_DIRS for part in p.parts)]
    return "\n".join(entries) or "(empty)"


def edit_file(path: str, old_str: str, new_str: str) -> str:
    """Replace old_str with new_str. old_str must match EXACTLY once.
    With old_str="" on a file that doesn't exist yet, creates it with new_str."""
    p = WORKSPACE / path
    if not p.exists():
        if old_str == "":                              # Ball's trick: empty anchor = create the file
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text(new_str)
            return f"Created {path}"
        return f"Error: {path} does not exist"
    text = p.read_text()
    count = text.count(old_str)
    if count == 0:
        return "Error: old_str not found. Read the file again — your copy may be stale."
    if count > 1:
        return f"Error: old_str matches {count} places. Include more surrounding lines so it is unique."
    p.write_text(text.replace(old_str, new_str, 1))
    return f"Edited {path}"

# quick smoke test, no model involved:
print(read_file("stats.py", limit=8))

## The tools, part 2 — a shell

The shell is what makes a coding agent *general*: anything you'd type in a terminal, it can request. It's also the most dangerous tool, so it ships with three limits baked in — a **timeout** (a hung command must not hang the agent), **output truncation** (a chatty command must not flood the context window — same budget logic as `read_file`), and a fixed **working directory** (the workspace, nothing above it).

In [ ]:
import subprocess

def run_bash(command: str) -> str:
    """Run a shell command inside the workspace. Returns exit code + output."""
    try:
        r = subprocess.run(command, shell=True, cwd=WORKSPACE,
                           capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return "Error: command timed out after 30s"
    out = (r.stdout + r.stderr).strip()
    if len(out) > 3000:
        out = out[:3000] + f"\n... (truncated {len(out) - 3000} chars)"
    return f"exit code: {r.returncode}\n{out or '(no output)'}"

# see the bug with our own eyes before any agent touches it:
print(run_bash("python -m pytest -q"))

One test red, exactly as planned. Hold that output in mind — it's the "before" picture.

Now the schemas. Same shape as 0.3, one entry per tool. Notice how much work the `description` fields do: they say *when* to reach for the tool and what its failure modes mean, because the description is the only manual the model gets.

In [ ]:
TOOL_FUNCTIONS = {
    "read_file": read_file, "list_files": list_files,
    "edit_file": edit_file, "run_bash": run_bash,
}

tools = [
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read a file from the workspace, with line numbers. Always read a file before editing it.",
        "parameters": {"type": "object", "properties": {
            "path":   {"type": "string",  "description": "Relative path, e.g. stats.py"},
            "offset": {"type": "integer", "description": "First line to read (default 1)"},
            "limit":  {"type": "integer", "description": "Max lines to return (default 200)"},
        }, "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "list_files",
        "description": "List all files in the workspace. Use it first to see what exists.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "Subdirectory to list (default: workspace root)"},
        }}}},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": ("Edit a file by replacing old_str (must match exactly once, copy it verbatim "
                        "from read_file output without the line numbers) with new_str. "
                        "To create a new file, pass old_str=\"\"."),
        "parameters": {"type": "object", "properties": {
            "path":    {"type": "string"},
            "old_str": {"type": "string", "description": "Exact text to replace — include enough lines to be unique"},
            "new_str": {"type": "string", "description": "The replacement text"},
        }, "required": ["path", "old_str", "new_str"]}}},
    {"type": "function", "function": {
        "name": "run_bash",
        "description": ("Run a shell command in the workspace (30s timeout). "
                        "Use 'python -m pytest -q' to run the tests and verify your changes."),
        "parameters": {"type": "object", "properties": {
            "command": {"type": "string"},
        }, "required": ["command"]}}},
]
print(f"{len(tools)} tools described")

## The permission gate

Left alone, the loop would execute whatever the model asks, the moment it asks. No real coding agent works that way. The cleanest mental model is Codex CLI's, which separates two questions:

| Question | Mechanism | In Codex CLI | Here |
|---|---|---|---|
| What *can* the agent technically touch? | **Sandbox** | OS-level (Seatbelt on macOS, Landlock on Linux), network off by default | every tool is jailed to `WORKSPACE` |
| When must it *ask* before acting? | **Approval policy** | `untrusted` / `on-request` / `never` | the function below |

Our policy, which is also roughly Claude Code's default: **reads are free, writes ask** — unless the command matches a pre-approved allowlist. Watch what that buys: the agent can explore all it wants without nagging you, but nothing changes on disk and no unexpected command runs without a human seeing it first.

In [ ]:
READ_ONLY = {"read_file", "list_files"}

# commands that may run without asking (prefix match — the test runner is the big one):
ALLOWED_BASH_PREFIXES = ("python -m pytest", "pytest", "ls", "cat ", "python --version")

def approve(name: str, args: dict) -> bool:
    """The approval policy: reads are free, writes and unknown commands ask."""
    if name in READ_ONLY:
        return True
    if name == "run_bash":
        cmd = args.get("command", "")
        if cmd.startswith(ALLOWED_BASH_PREFIXES):
            return True
        return input(f"  [approve?] run_bash: {cmd!r}  (y/n) ").strip().lower() == "y"
    if name == "edit_file":
        return input(f"  [approve?] edit_file: {args.get('path')}  (y/n) ").strip().lower() == "y"
    return False    # unknown tool -> deny by default

## Environment context: what the agent knows before turn one

Type a task into Claude Code and, before the model sees your words, the harness has already injected: the working directory, the platform, the file tree, `git status`, and the whole of `CLAUDE.md`. That's why it never asks "what files does this project have?" — the answer was in the system prompt all along.

We do the same. Note the rules at the bottom: read-before-edit and verify-after-change are the two habits that separate agents that work from agents that flail.

In [ ]:
import platform

def build_system_prompt() -> str:
    conventions = (WORKSPACE / "AGENTS.md").read_text() if (WORKSPACE / "AGENTS.md").exists() else "(none)"
    return f"""You are a careful coding agent working in a sandboxed workspace.

Environment:
- platform: {platform.system()}
- files in the workspace:
{list_files(".")}

Project conventions (AGENTS.md):
{conventions}

Rules:
- Always read a file before editing it.
- After changing code, run the tests to verify the change actually worked.
- Keep edits minimal: change what the task requires and nothing else.
- If a tool returns an error, read it carefully and adjust — do not repeat the same call.
"""

print(build_system_prompt())

## The loop — 0.3, plus the gate

This is `run_agent` from 0.3 with three additions, each one line of idea: the system prompt is *built* (environment injection), every tool call passes through `approve()` (a denial becomes an observation the model can react to), and the tally now prints per turn so you can watch the budget move.

In [ ]:
def coding_agent(task: str, max_turns: int = 15) -> str:
    messages = [{"role": "system", "content": build_system_prompt()},
                {"role": "user", "content": task}]
    total_tokens = 0

    for turn in range(1, max_turns + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, temperature=0,
        )
        total_tokens += resp.usage.total_tokens
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:                                # no tool wanted -> done
            print(f"  [cost] {total_tokens} tokens across {turn} turns")
            return msg.content

        for tc in msg.tool_calls:
            args = tc.function.arguments                      # raw string, kept for the log
            try:
                args = json.loads(args)
                if approve(tc.function.name, args):           # <- the gate
                    result = TOOL_FUNCTIONS[tc.function.name](**args)
                else:
                    result = "Denied by user."                # a denial is just another observation
            except Exception as e:
                result = f"Error: {e}"
            shown = str(args)[:80]
            print(f"  [tool] {tc.function.name}({shown}) -> {str(result)[:100]!r}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

    return "Stopped: reached max turns."

## First run: read-only

Start with a task that needs no approvals, to see the exploration behaviour on its own:

In [ ]:
print(coding_agent("Read stats.py and explain in two sentences what it provides and whether you see any bug."))

The agent listed nothing and asked nothing — it went straight to `read_file("stats.py")`, because the file tree was already in its system prompt. Environment injection pays for itself on the very first turn. (Whether it *spotted* the bug varies by model — which is exactly why the next run doesn't rely on the model's eyes, but on the tests.)

## The real thing: fix the failing tests

Now the task every coding agent demo earns its keep on. You'll be asked to approve the edit (answer `y`); the pytest runs are pre-approved by the allowlist. Watch the loop: **run tests → read code → edit → run tests again**. That last step is the verification loop — the agent doesn't claim the bug is fixed, it *proves* it, against a check it can run.

In [ ]:
print(coding_agent(
    "The test suite has a failing test. Find the bug, fix it, and run the tests to confirm they all pass."
))

In [ ]:
# Trust, but verify — OUTSIDE the agent. The same check, run by you:
print(run_bash("python -m pytest -q"))
print(read_file("stats.py", offset=6, limit=8))

All green, and the `median` now averages the two middle values on even-length input. Two things worth savoring:

**The agent never saw "the answer."** Nobody told it which test failed or what the fix was. It ran pytest, read the assertion error, read the code, formed the fix, and re-ran pytest. The failing test did the teaching. This is why Anthropic's first best-practice for Claude Code is *give it a check it can run* — with ground truth available, even a mid-tier model converges; without it, even a frontier model just makes confident claims.

**Your final verification ran outside the agent.** The `run_bash` above wasn't a tool call — it was you. An agent's own "all tests pass" is a model output like any other; the habit of re-checking with your own hands (or your CI) is what 0.5 called putting the gate around the irreversible step.

::::{dropdown} 🔍 What actually happened, turn by turn
:color: info

A typical successful run of the fix task (yours may vary in order — that's the agent deciding):

```
turn 1  run_bash("python -m pytest -q")        -> exit 1, "test_median_even FAILED ... assert 3 == 2.5"
turn 2  read_file("stats.py")                  -> sees median() indexing the middle element
turn 3  edit_file("stats.py",
          old_str="    ordered = sorted(numbers)\n    return ordered[len(ordered) // 2]",
          new_str="    ordered = sorted(numbers)\n    n = len(ordered)\n    mid = n // 2\n"
                  "    if n % 2 == 1:\n        return ordered[mid]\n"
                  "    return (ordered[mid - 1] + ordered[mid]) / 2")
                                               -> "Edited stats.py"   (after your y)
turn 4  run_bash("python -m pytest -q")        -> exit 0, "3 passed"
turn 5  no tool_calls -> final answer
```

Map it back to the message list from 0.3: every arrow is one append. The pytest *output text* — `assert 3 == 2.5` — is what steered turn 2. Feedback-driven, literally.
::::

## What the real ones add

You now have the skeleton shared by every production coding agent. Here is where each real tool goes further — and notice they all answer the *same five questions* differently:

| | **Claude Code** | **Codex CLI** | **Aider** | **OpenCode** |
|---|---|---|---|---|
| **Find code** | `Grep`/`Glob` (ripgrep) — regex search, no embeddings | search + file reads | **repo-map**: tree-sitter parses every file, a PageRank-style ranking picks which symbols fit the token budget | grep/glob |
| **Edit files** | string-replace with unique anchor (ours) | `apply_patch` with the **V4A** patch format — GPT models are *trained* on it | multiple diff formats, chosen per model | string-replace |
| **Stay safe** | permission prompts + allowlists + hooks | **OS sandbox** (Seatbelt/Landlock, network off) × approval policy | auto-commits to git — undo is `git revert` | per-tool `allow`/`ask`/`deny` rules |
| **Survive long sessions** | compaction + subagents (next notebook, 0.8) | compaction | repo-map keeps context small from the start | compaction + subagents |
| **Learn the project** | `CLAUDE.md` | `AGENTS.md` | conventions files | `AGENTS.md` |

Two rows deserve numbers:

**Editing.** The edit format is not a detail — Aider measured it. Switching GPT-4 Turbo from whole-file rewrites to unified diffs took its benchmark success rate **from 20% to 61%**, and applying patches *flexibly* (tolerating slightly-off line positions) cut failed edits by ~9× ([aider.chat on unified diffs](https://aider.chat/2023/12/21/unified-diffs.html)). Our unique-anchor rule is the notebook-sized version of the same insight: make the edit operation hard to get wrong.

**Search.** Claude Code deliberately uses regex search instead of RAG over the codebase. The [MinusX teardown](https://minusx.ai/blog/decoding-claude-code/) argues why: embedding-based retrieval over code has silent failure modes (chunking splits functions, similarity misses exact identifiers), while ripgrep + the model iterating on queries is transparent and debuggable. Aider's repo-map is the strongest counter-example — precomputed structure instead of on-demand search. Both work; both are honest about their trade-off.

::::{dropdown} 🔧 Under the hood: what else is in the harness
:color: info

Things the real harnesses do that we skipped, worth knowing by name:

- **System prompt scale.** Claude Code's system prompt is ~2,800 tokens, plus ~9,400 tokens of tool descriptions — a reminder that tool descriptions are load-bearing (1.2 makes this the whole lesson).
- **Model routing.** More than half of Claude Code's LLM calls go to a *small* model (Haiku) — parsing bash output, summarizing, classifying — not the frontier model. Cost engineering, not intelligence engineering.
- **Hooks.** Deterministic scripts that run before/after tool calls (block an edit to `prod.env`, run a linter after every edit). Guardrails as code, not prompts.
- **Checkpoints.** Snapshot files before each edit so any change can be rewound — the harness-level undo that makes auto-accept modes tolerable.
- **Plan mode.** A read-only mode where the agent explores and proposes before touching anything — the HITL gate of 0.5 promoted to a workflow stage.

For the full picture from primary sources: [How Claude Code works](https://code.claude.com/docs/en/how-claude-code-works), [Codex sandboxing](https://developers.openai.com/codex/concepts/sandboxing), and [12-Factor Agents](https://github.com/humanlayer/12-factor-agents) — the best "principles" write-up of this whole space.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a search tool.** Write `search_files(pattern: str)` that greps every workspace file and returns `file:line: text` matches, register it (function, schema, `READ_ONLY`), and ask the agent: *"Which test exercises the even-length case, and which function does it call?"* **Done when:** the agent answers via `search_files` without reading whole files.
2. **Harden the gate.** Make `approve()` deny any `run_bash` whose command contains `rm `, `sudo` or `>` outright (no prompt), and add `"python "` to the allowlist. Then ask the agent to *"delete the test file and make the suite pass"* and watch the gate + `AGENTS.md` conventions hold the line. **Done when:** `test_stats.py` still exists and the agent explains why it didn't delete it (or its deletion attempt shows as `Denied`).
3. **Teach it a convention.** Add `- All docstrings must be written in Spanish.` to `AGENTS.md`, then ask the agent to add a `variance(numbers)` function with tests. **Done when:** the new docstring comes out in Spanish without the task mentioning language — proof that environment context steers behaviour.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Search tool:**

```python
def search_files(pattern: str) -> str:
    """Search all workspace files for a text pattern."""
    hits = []
    for p in sorted(WORKSPACE.rglob("*")):
        if p.is_file():
            for n, line in enumerate(p.read_text(errors="ignore").splitlines(), 1):
                if pattern in line:
                    hits.append(f"{p.relative_to(WORKSPACE)}:{n}: {line.strip()}")
    return "\n".join(hits[:50]) or "no matches"

TOOL_FUNCTIONS["search_files"] = search_files
READ_ONLY.add("search_files")
tools.append({"type": "function", "function": {
    "name": "search_files",
    "description": "Search every workspace file for a text pattern. Returns file:line: text matches. Cheaper than reading whole files when you only need to locate something.",
    "parameters": {"type": "object", "properties": {
        "pattern": {"type": "string"}}, "required": ["pattern"]}}})
```

This is Claude Code's `Grep` in miniature. Note the description *sells the trade-off* ("cheaper than reading whole files") — that's what makes the model prefer it when appropriate.

**2 — Harden the gate:**

```python
BLOCKED_SUBSTRINGS = ("rm ", "sudo", ">")

def approve(name, args):
    if name in READ_ONLY:
        return True
    if name == "run_bash":
        cmd = args.get("command", "")
        if any(b in cmd for b in BLOCKED_SUBSTRINGS):
            return False                                  # hard deny, no human needed
        if cmd.startswith(ALLOWED_BASH_PREFIXES + ("python ",)):
            return True
        return input(f"  [approve?] run_bash: {cmd!r}  (y/n) ").strip().lower() == "y"
    if name == "edit_file":
        return input(f"  [approve?] edit_file: {args.get('path')}  (y/n) ").strip().lower() == "y"
    return False
```

Three tiers — always-deny, always-allow, ask — is exactly the shape of Codex's approval policies and OpenCode's `allow`/`ask`/`deny`. (A substring blocklist is demo-grade, not security: 1.5 returns to why, and real sandboxes work at the OS level precisely because string-matching can be dodged.)

**3 — Convention:** no code needed — append the line to `AGENTS.md` and run the task. The system prompt is rebuilt on every `coding_agent()` call, so the new convention is present from turn one. If the docstring still comes out in English, check that your task didn't override it ("add a variance function *with an English docstring*" beats the conventions file — instructions closer to the task win).
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| `edit_file` keeps returning "old_str not found" | The model copied line numbers (`   7\| ...`) into `old_str`, or its copy of the file is stale after a previous edit | The error message already tells it to re-read; if it thrashes, sharpen the schema description ("without the line numbers") |
| "old_str matches 2 places" loops | The anchor is too short (e.g. a bare `return` line) | That's the tool working as designed — the model should include more surrounding lines; if it doesn't, say so in the tool description |
| The agent claims success but tests still fail | It skipped the verify step | Tighten the rule in the system prompt ("run the tests *after every edit*"), or refuse to accept a final answer until `run_bash` shows exit code 0 — you can enforce that in the loop |
| `pytest: command not found` | pytest missing or PATH issues in the shell | Use `python -m pytest` form (the allowlist and prompts here already do) |
| The loop hits `max_turns` re-reading files | Tool results too large, drowning the plan (context rot) | Lower `read_file`'s default `limit`, truncate harder in `run_bash` — and see 0.8, which is entirely about this failure mode |
| `input()` prompt never appears | Some notebook frontends buffer it while a cell streams output | Run in Colab/Jupyter proper; or temporarily auto-approve by returning `True` (accepting the risk that implies) |
::::

**What's next:** your agent works — until the session gets long. Big files, verbose test output and dozens of turns will eventually drown the plan in noise and blow the context window. **0.8** is about the discipline the real harnesses obsess over: **context engineering** — compaction, subagents, and plans that survive.